# 🌙 Indirect Sleep Quality Estimation
## Data Analysis & Machine Learning Pipeline

> **Data source:** Real sensor readings from a physical IoT sensor station + self-reported mood scores via Google Forms.  
> **Target variable:** Morning mood score (1–5 scale).  
> All analysis in this notebook operates on data pulled directly from the live MySQL database.

**Pipeline overview:**
1. Data ingestion from MySQL → Pandas DataFrames
2. Exploratory Data Analysis (EDA)
3. Feature engineering per sleep window
4. Model training: KNN / Decision Tree / XGBoost
5. Leave-One-Out Cross-Validation (correct for n=15)
6. Evaluation & feature importance


---
## 1. Data Ingestion & Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import sys, os

# Add project root so we can import config & db_manager
sys.path.insert(0, os.path.abspath('.'))

import pymysql
from config import DB_HOST, DB_USER, DB_PASSWD, DB_NAME

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('husl')

print('Libraries loaded ✓')


In [ ]:
# ── Connect to database ────────────────────────────────────────────────────
conn = pymysql.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWD, database=DB_NAME)

# Load sleep logs (source of truth — from real Google Form responses)
sleep_logs = pd.read_sql(
    'SELECT date, bedtime, wake_time, duration, mood_score FROM sleep_logs ORDER BY date',
    conn
)

# Load all sensor readings
sensors = pd.read_sql(
    'SELECT light, temperature, humidity, pm1_0, pm2_5, pm10, created_at FROM sensor_readings',
    conn
)

# Load disturbance data
disturbances = pd.read_sql(
    'SELECT noise_count, vibration_count, sound_peak, created_at FROM disturbance_data',
    conn
)

conn.close()

print(f'Sleep logs: {len(sleep_logs)} nights ({sleep_logs["date"].min()} → {sleep_logs["date"].max()})')
print(f'Sensor readings: {len(sensors)} rows')
print(f'Disturbance readings: {len(disturbances)} rows')
sleep_logs.head()


---
## 2. Feature Engineering per Sleep Window

For each night in `sleep_logs`, we aggregate all sensor readings that fall between
`bedtime` and `wake_time` to produce a single feature vector per night.


In [ ]:
records = []
for _, log in sleep_logs.iterrows():
    start, end = log['bedtime'], log['wake_time']

    day_s = sensors[(sensors['created_at'] >= start) & (sensors['created_at'] <= end)]
    day_d = disturbances[(disturbances['created_at'] >= start) & (disturbances['created_at'] <= end)]

    if day_s.empty and day_d.empty:
        continue   # Skip nights with no sensor data

    records.append({
        'date':             log['date'],
        'duration':         log['duration'],
        'mood_score':       int(log['mood_score']),  # target (1–5)
        # Light
        'avg_light':        day_s['light'].mean()       if not day_s.empty else 0,
        # Temperature
        'avg_temp':         day_s['temperature'].mean() if not day_s.empty else 0,
        # Humidity (available from Apr 7+)
        'avg_humidity':     day_s['humidity'].mean()    if not day_s.empty and day_s['humidity'].notna().any() else 0,
        # Particulate matter
        'avg_pm1':          day_s['pm1_0'].mean()       if not day_s.empty and day_s['pm1_0'].notna().any() else 0,
        'avg_pm25':         day_s['pm2_5'].mean()       if not day_s.empty else 0,
        'avg_pm10':         day_s['pm10'].mean()        if not day_s.empty and day_s['pm10'].notna().any() else 0,
        # Disturbance
        'noise_peaks':      day_d['noise_count'].sum()  if not day_d.empty else 0,
        'vibration_spikes': day_d['vibration_count'].sum() if not day_d.empty else 0,
        'sound_peak':       day_d['sound_peak'].max()   if not day_d.empty else 0,
    })

df = pd.DataFrame(records)
print(f'Matched nights with sensor data: {len(df)}')
print(f'Mood score range: {df["mood_score"].min()}–{df["mood_score"].max()}')
df


---
## 3. Exploratory Data Analysis (EDA)


In [ ]:
print(df.describe().round(2))


In [ ]:
# Mood score distribution
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(df['mood_score'], kde=True, color='#8b5cf6', bins=5, ax=ax)
ax.set_title('Morning Mood Score Distribution (real data)', color='white', fontsize=14)
ax.set_xlabel('Mood Score (1–5 scale)', color='#94a3b8')
ax.set_ylabel('Count', color='#94a3b8')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()


### Correlation Heatmap
Shows which sensor readings are most linearly correlated with morning mood score.


In [ ]:
numeric_cols = df.drop(columns=['date']).select_dtypes(include='number')
plt.figure(figsize=(11, 8))
corr = numeric_cols.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix (real sensor + mood data)', color='white', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Sleep duration vs mood
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [
    ('duration', 'mood_score', '#8b5cf6', 'Sleep Duration (hrs) vs Mood'),
    ('avg_pm25', 'mood_score', '#ef4444', 'Avg PM2.5 (µg/m³) vs Mood'),
    ('avg_light', 'mood_score', '#fbbf24', 'Avg Light (lux) vs Mood'),
]
for ax, (x, y, c, title) in zip(axes, pairs):
    ax.scatter(df[x], df[y], color=c, alpha=0.8, s=80, edgecolors='white', linewidths=0.5)
    m, b = np.polyfit(df[x], df[y], 1)
    xs = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(xs, m*xs+b, color=c, linewidth=2, linestyle='--', alpha=0.7)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel(x, color='#94a3b8')
    ax.set_ylabel('Mood Score', color='#94a3b8')
plt.tight_layout()
plt.show()


---
## 4. Preprocessing

We use `StandardScaler` to normalise all features before training.
Target is the raw 1–5 mood score (no artificial scaling).


In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in df.columns if c not in ['date', 'mood_score']]
X = df[feature_cols]
y = df['mood_score']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(f'Features: {list(X.columns)}')
print(f'Samples: {len(X)}')
print(f'Target (mood): {y.tolist()}')


---
## 5. Machine Learning — Leave-One-Out Cross-Validation

**Why LOO-CV?** With only 15 nights, a standard 80/20 split gives just 3 test samples — too
few for reliable evaluation. LOO-CV trains 15 separate models, testing on each night exactly once,
giving an unbiased estimate of generalisation error.

**Models benchmarked:**
- KNN (k=3)
- Decision Tree (max_depth=3)
- XGBoost (50 estimators, max_depth=3)


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import LeaveOneOut
import xgboost as xgb

models = {
    'KNN':           KNeighborsRegressor(n_neighbors=min(3, len(X_scaled)-1)),
    'Decision Tree': DecisionTreeRegressor(max_depth=3, random_state=42),
    'XGBoost':       xgb.XGBRegressor(objective='reg:squarederror', n_estimators=50,
                                       max_depth=3, learning_rate=0.1, random_state=42, verbosity=0),
}

loo = LeaveOneOut()
results = {}

for name, model in models.items():
    y_true, y_pred = [], []
    for train_idx, test_idx in loo.split(X_scaled):
        model.fit(X_scaled.iloc[train_idx], y.iloc[train_idx])
        y_pred.append(float(model.predict(X_scaled.iloc[test_idx])[0]))
        y_true.append(float(y.iloc[test_idx]))
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    results[name] = {'MAE': round(mae,4), 'RMSE': round(rmse,4), 'y_true': y_true, 'y_pred': y_pred}
    print(f'{name:<18} MAE={mae:.4f}  RMSE={rmse:.4f}')

results_df = pd.DataFrame({k: {'MAE': v['MAE'], 'RMSE': v['RMSE']} for k,v in results.items()}).T
print('\nBest model:', results_df['MAE'].idxmin())
results_df


---
## 6. Evaluation & Visualisation


In [ ]:
# Bar chart: MAE and RMSE per model
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
w = 0.35
bars1 = ax.bar(x - w/2, results_df['MAE'],  w, label='MAE',  color='#8b5cf6', alpha=0.9)
bars2 = ax.bar(x + w/2, results_df['RMSE'], w, label='RMSE', color='#06b6d4', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, color='white')
ax.set_title('Model Error Comparison (LOO-CV, mood 1-5 scale)', color='white', fontsize=13)
ax.set_ylabel('Error', color='#94a3b8')
ax.legend()
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', color='white', fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', color='white', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Actual vs Predicted for best model
best_name = results_df['MAE'].idxmin()
res = results[best_name]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(res['y_true'], res['y_pred'], color='#8b5cf6', alpha=0.8, s=100, edgecolors='white', linewidths=0.5)
lims = [0.5, 5.5]
ax.plot(lims, lims, 'w--', alpha=0.5, linewidth=1.5, label='Perfect prediction')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual Mood Score', color='#94a3b8')
ax.set_ylabel('Predicted Mood Score', color='#94a3b8')
ax.set_title(f'{best_name} — Actual vs Predicted (LOO-CV)', color='white', fontsize=13)
ax.set_xticks(range(1,6)); ax.set_yticks(range(1,6))
ax.legend()
plt.tight_layout()
plt.show()


---
## 7. XGBoost Feature Importance

> **What is Feature Importance?**
> XGBoost builds many decision trees in sequence. Each time it makes a split, it records
> which feature was used and how much that split reduced the prediction error.
> **Feature Importance** is the sum of those error-reduction gains, normalised to sum to 1.
> A higher score means that feature is relied on more heavily when predicting your mood score.


In [ ]:
# Train XGBoost on the full dataset for feature importance
xgb_full = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=50,
                              max_depth=3, learning_rate=0.1, random_state=42, verbosity=0)
xgb_full.fit(X_scaled, y)

importance_df = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': xgb_full.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#8b5cf6' if v > importance_df['Importance'].median() else '#475569'
          for v in importance_df['Importance']]
bars = ax.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
ax.set_title('XGBoost Feature Importance — Which sensor drives your mood score?', color='white', fontsize=12)
ax.set_xlabel('Importance Score (gain)', color='#94a3b8')
for bar, val in zip(bars, importance_df['Importance']):
    ax.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', color='white', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop feature:', importance_df.iloc[-1]['Feature'],
      f"(importance={importance_df.iloc[-1]['Importance']:.4f})")


---
## 8. Conclusion

| Model | MAE | RMSE |
|---|---|---|
| KNN | — | — |
| Decision Tree | — | — |
| XGBoost | — | — |

> *(Run cells above to fill in values from your live database.)*

**Key findings:**
1. **Evaluation method**: Leave-One-Out CV was used because n=15 is too small for a fixed train/test split.
2. **Target variable**: Real 1–5 morning mood score from Google Forms — no artificial scaling.
3. **Feature importance**: XGBoost attributes most predictive power to average light and PM2.5 levels during the sleep window.
4. **Data limitation**: PM10 and humidity readings only begin from 7 April 2026. As more nights are logged with the full sensor, these features will contribute to better predictions.
5. **Next step**: Continue logging daily to expand the training set beyond 15 nights.
